# POMDP Candidate Evaluation

This notebook evaluates the belief-limited attacker over the mixed `70/30` VAE candidate pool. It compares `good_contact`, `baseline_night`, and `poor_contact` across multiple observation seeds so one noisy draw does not overstate or hide preset effects.

The selector only sees noisy attacker-facing observations and candidate parameters. It does not use true hit counts, expected loss, or full-state ranks while selecting candidates.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Resolve repo root when this notebook is launched from either repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "convoy_sim").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from convoy_sim.realism import get_attacker_observation_config
from experiments.evaluate_attack_candidate_pool import evaluate_candidate_pool
from experiments.run_pomdp_candidate_selector import run_belief_selector


## Configuration

`good_contact` is the default realism preset. The full-state run path points at the latest mixed-VAE final baseline run, so the notebook can compute top-k overlap without rerunning the expensive oracle selector.


In [ ]:
CANDIDATE_PATH = PROJECT_ROOT / "data" / "attack_profiles" / "vae_candidates" / "mixed_curated70_random30_hit_candidates.jsonl"
FULL_STATE_RUN_DIR = PROJECT_ROOT / "results" / "runs" / "candidate_pool_eval" / "20260512_144030_vae_final_baseline_mixed_vae"

OBSERVATION_PRESETS = ["good_contact", "baseline_night", "poor_contact"]
OBSERVATION_SEEDS = [1945, 1946, 1947, 1948, 1949]
MAX_PROFILES = 1000
TOP_K = 25

RUN_SELECTION = True
RUN_MONTE_CARLO_EVAL = True

CONVOY_PROFILE = "convoy_layout_1"
EVAL_SEEDS = [1942, 1943, 1944]
N_TRIALS_PER_SEED = 10
T_MAX = 400.0
OBJECTIVE_CFG = {"preset": "balanced_default"}

OUTPUT_ROOT = Path("results/runs")


In [ ]:
preset_rows = []
for preset in OBSERVATION_PRESETS:
    cfg = get_attacker_observation_config(preset)
    row = {"preset": preset}
    row.update(cfg.to_dict())
    preset_rows.append(row)

pd.DataFrame(preset_rows)


## Belief-Limited Selection

This ranks the mixed VAE candidate pool under every observation preset and observation seed. Each run writes a top-k JSONL candidate pool that can be passed directly into the existing Monte Carlo evaluator.


In [ ]:
selection_runs: list[dict] = []

if RUN_SELECTION:
    for preset in OBSERVATION_PRESETS:
        for observation_seed in OBSERVATION_SEEDS:
            run_dir = run_belief_selector(
                candidate_path=CANDIDATE_PATH,
                project_root=PROJECT_ROOT,
                output_root=OUTPUT_ROOT,
                run_name=f"pomdp_{preset}_seed{observation_seed}_notebook",
                convoy_profile=CONVOY_PROFILE,
                max_profiles=MAX_PROFILES,
                top_k=TOP_K,
                seed=int(observation_seed),
                observation_preset=preset,
            )
            selection_runs.append({
                "preset": preset,
                "observation_seed": int(observation_seed),
                "selection_dir": run_dir,
            })
            print(f"{preset} seed={observation_seed}: {run_dir}")
else:
    selection_runs = [
        # {"preset": "good_contact", "observation_seed": 1945, "selection_dir": PROJECT_ROOT / "results/runs/pomdp_candidate_selector/<existing_run>"},
    ]

pd.DataFrame(selection_runs)


In [ ]:
belief_tables = []
for item in selection_runs:
    df = pd.read_csv(item["selection_dir"] / "belief_ranked_candidates.csv")
    df.insert(0, "observation_seed", int(item["observation_seed"]))
    df.insert(0, "preset", str(item["preset"]))
    belief_tables.append(df)

belief_df = pd.concat(belief_tables, ignore_index=True) if belief_tables else pd.DataFrame()
belief_df.head(10)


## Evaluate Selected Top-K Candidates

Each selected top-k pool is evaluated with the same scored Monte Carlo pipeline used by the full-state attacker baseline.


In [ ]:
eval_runs: list[dict] = []

if RUN_MONTE_CARLO_EVAL:
    for item in selection_runs:
        preset = str(item["preset"])
        observation_seed = int(item["observation_seed"])
        selected_pool = item["selection_dir"] / "top_belief_candidate_pool.jsonl"
        run_dir = evaluate_candidate_pool(
            candidate_path=selected_pool,
            project_root=PROJECT_ROOT,
            output_root=OUTPUT_ROOT,
            run_name=f"pomdp_{preset}_seed{observation_seed}_top{TOP_K}_eval_notebook",
            convoy_profile=CONVOY_PROFILE,
            max_profiles=None,
            top_k=TOP_K,
            seeds=EVAL_SEEDS,
            n_trials_per_seed=N_TRIALS_PER_SEED,
            t_max=T_MAX,
            max_hits_per_torpedo=1,
            objective_cfg=OBJECTIVE_CFG,
        )
        eval_runs.append({
            "preset": preset,
            "observation_seed": observation_seed,
            "eval_dir": run_dir,
        })
        print(f"{preset} seed={observation_seed}: {run_dir}")
else:
    eval_runs = [
        # {"preset": "good_contact", "observation_seed": 1945, "eval_dir": PROJECT_ROOT / "results/runs/candidate_pool_eval/<existing_run>"},
    ]

pd.DataFrame(eval_runs)


## Per-Run Summary

For POMDP rows, `candidate_pool` is the belief-selected top-k pool. For the optional full-state row, `top_k` is the oracle-selected top-k from the existing full-state mixed-VAE baseline.


In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

summary_rows = []

if FULL_STATE_RUN_DIR.exists():
    metrics = load_json(FULL_STATE_RUN_DIR / "metrics_summary.json")
    top = metrics["top_k"]
    best = metrics.get("best_candidate") or {}
    summary_rows.append({
        "selector": "full_state_oracle",
        "preset": "oracle",
        "observation_seed": -1,
        "profiles": int(top.get("profiles", TOP_K)),
        "expected_hits": float(top["expected_hits"]),
        "expected_loss": float(top.get("expected_loss", 0.0)),
        "expected_unique_ships_hit": float(top.get("expected_unique_ships_hit", 0.0)),
        "CVaR_90": float(top.get("CVaR_90", 0.0)),
        "CVaR_90_loss": float(top.get("CVaR_90_loss", 0.0)),
        "best_profile_id": str(best.get("profile_id", "")),
    })

for item in eval_runs:
    metrics = load_json(item["eval_dir"] / "metrics_summary.json")
    pool = metrics["candidate_pool"]
    best = metrics.get("best_candidate") or {}
    summary_rows.append({
        "selector": "belief_limited",
        "preset": str(item["preset"]),
        "observation_seed": int(item["observation_seed"]),
        "profiles": int(pool.get("profiles", TOP_K)),
        "expected_hits": float(pool["expected_hits"]),
        "expected_loss": float(pool.get("expected_loss", 0.0)),
        "expected_unique_ships_hit": float(pool.get("expected_unique_ships_hit", 0.0)),
        "CVaR_90": float(pool.get("CVaR_90", 0.0)),
        "CVaR_90_loss": float(pool.get("CVaR_90_loss", 0.0)),
        "best_profile_id": str(best.get("profile_id", "")),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


## Preset Aggregate

This is the main comparison table. Means and standard deviations are over observation seeds; Monte Carlo seeds/trials are the same across rows.


In [ ]:
metric_cols = ["expected_hits", "expected_loss", "expected_unique_ships_hit", "CVaR_90", "CVaR_90_loss"]

pomdp_df = summary_df.loc[summary_df["selector"] == "belief_limited"].copy()
agg_df = (
    pomdp_df.groupby("preset", as_index=False)[metric_cols]
    .agg(["mean", "std", "min", "max"])
)
agg_df.columns = ["preset" if col[0] == "preset" else f"{col[0]}_{col[1]}" for col in agg_df.columns]
agg_df = agg_df.reset_index(drop=True)

if FULL_STATE_RUN_DIR.exists():
    oracle_row = summary_df.loc[summary_df["selector"] == "full_state_oracle"].iloc[0]
    for metric in metric_cols:
        agg_df[f"oracle_gap_{metric}"] = float(oracle_row[metric]) - agg_df[f"{metric}_mean"]

agg_df


In [ ]:
if len(agg_df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="white")
    axes[0].bar(agg_df["preset"], agg_df["expected_hits_mean"], yerr=agg_df["expected_hits_std"], color="#4c78a8", capsize=4)
    axes[0].set_title("Expected Hits by Observation Preset")
    axes[0].set_ylabel("mean +/- std")
    axes[1].bar(agg_df["preset"], agg_df["expected_loss_mean"], yerr=agg_df["expected_loss_std"], color="#f58518", capsize=4)
    axes[1].set_title("Expected Loss by Observation Preset")
    axes[1].set_ylabel("mean +/- std")
    plt.tight_layout()
    plt.show()


## Rank Overlap Diagnostics

This checks whether noisy belief selection is choosing the same candidates as the full-state Monte Carlo oracle, and whether presets are selecting meaningfully different top-k sets.


In [ ]:
overlap_rows = []
if FULL_STATE_RUN_DIR.exists() and selection_runs:
    full_ranked = pd.read_csv(FULL_STATE_RUN_DIR / "ranked_candidates.csv")
    oracle_top = set(full_ranked.head(TOP_K)["profile_id"].astype(str))
    for item in selection_runs:
        belief_ranked = pd.read_csv(item["selection_dir"] / "belief_ranked_candidates.csv")
        belief_top = set(belief_ranked.head(TOP_K)["profile_id"].astype(str))
        overlap_rows.append({
            "preset": str(item["preset"]),
            "observation_seed": int(item["observation_seed"]),
            "top_k": TOP_K,
            "oracle_overlap_count": len(oracle_top & belief_top),
            "oracle_overlap_rate": len(oracle_top & belief_top) / float(TOP_K),
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df


In [ ]:
if len(overlap_df):
    overlap_agg = (
        overlap_df.groupby("preset", as_index=False)["oracle_overlap_rate"]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )
    display(overlap_agg)

    fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
    ax.bar(overlap_agg["preset"], overlap_agg["mean"], yerr=overlap_agg["std"], color="#54a24b", capsize=4)
    ax.set_ylim(0, 1)
    ax.set_title("Top-K Overlap With Full-State Oracle")
    ax.set_ylabel("mean overlap rate +/- std")
    ax.set_xlabel("observation preset")
    plt.tight_layout()
    plt.show()


In [ ]:
preset_pair_rows = []
for observation_seed in OBSERVATION_SEEDS:
    seed_runs = [item for item in selection_runs if int(item["observation_seed"]) == int(observation_seed)]
    top_by_preset = {}
    for item in seed_runs:
        df = pd.read_csv(item["selection_dir"] / "belief_ranked_candidates.csv")
        top_by_preset[str(item["preset"])] = set(df.head(TOP_K)["profile_id"].astype(str))
    for idx, preset_a in enumerate(OBSERVATION_PRESETS):
        for preset_b in OBSERVATION_PRESETS[idx + 1:]:
            if preset_a not in top_by_preset or preset_b not in top_by_preset:
                continue
            overlap = len(top_by_preset[preset_a] & top_by_preset[preset_b])
            preset_pair_rows.append({
                "observation_seed": int(observation_seed),
                "preset_a": preset_a,
                "preset_b": preset_b,
                "overlap_count": overlap,
                "overlap_rate": overlap / float(TOP_K),
            })

preset_pair_overlap_df = pd.DataFrame(preset_pair_rows)
preset_pair_overlap_df
